# Stage 2 — Frozen CLIP Feature Extraction

Goal: load frozen CLIP, run it on a handful of real memes, and sanity-check that
the embeddings actually behave the way CLIP embeddings should — before we trust
this pipeline enough to precompute embeddings for the entire dataset.

In [ ]:
import sys
sys.path.append("..")

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.data import load_all
from src.features import CLIPFeatureExtractor, get_device

print("Device:", get_device())

clean_splits, images_root = load_all()
train_df = clean_splits["train"]

## 1. Load frozen CLIP and embed a small batch

In [ ]:
extractor = CLIPFeatureExtractor()

sample = train_df.sample(8, random_state=1).reset_index(drop=True)
images = [Image.open(os.path.join(images_root, p)).convert("RGB") for p in sample["img"]]
texts = list(sample["text"])

image_embeds = extractor.embed_images(images)
text_embeds = extractor.embed_texts(texts)

print("Image embeddings shape:", image_embeds.shape)  # expect [8, 512]
print("Text embeddings shape:", text_embeds.shape)     # expect [8, 512]
print("Image embedding norm (should be ~1.0, confirms L2 normalization):", image_embeds[0].norm().item())

## 2. Sanity check: does CLIP actually associate the right image with the right text?

Important nuance: CLIP was pretrained on natural image-caption pairs, and meme text
is *not* a caption of the image (that's the whole point of memes — the humor/hate
comes from the mismatch or juxtaposition). So we should NOT expect a perfect diagonal
here the way we would for, say, COCO captions. What we're checking for is a *general
tendency* — matched pairs scoring noticeably higher on average than random mismatches —
which confirms CLIP's shared embedding space is still doing something meaningful for
this domain, even if imperfect. This is exactly why we need our own fusion + classifier
on top, rather than just thresholding raw CLIP similarity.

In [ ]:
similarity_matrix = image_embeds @ text_embeds.T  # cosine similarity since both are L2-normalized

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(similarity_matrix.numpy(), cmap="viridis")
ax.set_xlabel("Text index")
ax.set_ylabel("Image index")
ax.set_title("Image-Text Cosine Similarity (frozen CLIP)")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("../outputs/clip_similarity_sanity_check.png", dpi=150)
plt.show()

diagonal_mean = similarity_matrix.diagonal().mean().item()
off_diagonal_mean = (similarity_matrix.sum() - similarity_matrix.diagonal().sum()) / (similarity_matrix.numel() - similarity_matrix.shape[0])
print(f"Mean matched-pair similarity (diagonal): {diagonal_mean:.4f}")
print(f"Mean mismatched-pair similarity (off-diagonal): {off_diagonal_mean.item():.4f}")
print(f"Matched pairs score higher on average: {diagonal_mean > off_diagonal_mean.item()}")

## 3. Visual check: look at a couple of samples alongside their similarity scores

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16,4))
for i, ax in enumerate(axes):
    ax.imshow(images[i])
    ax.set_title(f'"{texts[i][:40]}"\nself-sim: {similarity_matrix[i,i]:.3f}', fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("../outputs/clip_sample_check.png", dpi=150)
plt.show()

## 4. Conclusion

Frozen CLIP (`openai/clip-vit-base-patch32`) successfully embeds both memes' images and text into a shared 512-dimensional space, confirmed by:
- Correct output shapes ([8, 512] for both modalities) and proper L2 normalization (embedding norm = 1.0).
- Matched image-text pairs scored higher on average (0.274) than mismatched pairs (0.200), confirming CLIP's shared embedding space still carries meaningful signal on meme data — even though meme text is not a literal caption of its image, which is why we don't expect (or need) a perfect diagonal.

This confirms the frozen backbone is a solid foundation to build on. **Next: run `python src/extract_embeddings.py`** to precompute and cache embeddings for the full filtered dataset (train: 6,744 / validation: 831 / test: 2,408), so Stage 3-4 (fusion + classifier head) can train directly on cached arrays without repeatedly re-running CLIP.